# 03. Training protocol: the two-phase LR schedule (centerpiece)

**Goal:** implement and plot the schedule. The learning rate ramps up over the first 50 epochs (linear warm-up), then decays polynomially. The whole network trains the whole time; no part is frozen.

In [ ]:
# bootstrap: make course_utils + scripts importable from any working directory,
# use the inline backend so figures render, and regenerate the phantom if missing.
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO = _find_repo_root(Path.cwd())
for _p in (str(REPO), str(REPO / "scripts")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

DATA = REPO / "assets" / "data" / "Dataset999_Phantom"
PRE = REPO / "assets" / "precomputed"

if not (DATA / "imagesTr" / "PHANTOM_001_0000.nii.gz").exists():
    import generate_phantom
    generate_phantom.generate(REPO / "assets" / "data")

print("repo root:", REPO.name)


## Drive the real schedulers

`course_utils` ships the same two schedulers the lab's fine-tuning trainer uses. We step a dummy optimizer through 1000 epochs and record the learning rate.

In [ ]:
import torch
from course_utils.lr_schedulers import Lin_incr_LRScheduler, PolyLRScheduler_offset

MAX_LR, WARMUP, TOTAL = 1e-3, 50, 1000
param = torch.nn.Parameter(torch.zeros(1))
opt = torch.optim.SGD([param], lr=MAX_LR, momentum=0.99, nesterov=True)

warm = Lin_incr_LRScheduler(opt, max_lr=MAX_LR, max_steps=WARMUP)
poly = PolyLRScheduler_offset(opt, initial_lr=MAX_LR, max_steps=TOTAL, start_step=WARMUP)
lrs = []
for epoch in range(TOTAL):
    sched = warm if epoch < WARMUP else poly   # the trainer swaps schedulers at epoch 50
    sched.step(epoch)
    lrs.append(opt.param_groups[0]['lr'])

print('epoch   0:', round(lrs[0], 8), '(about 2e-5, not zero)')
print('epoch  49:', round(lrs[49], 8), '(peak)')
print('epoch  50:', round(lrs[50], 8), '(decay restarts at the peak)')
print('epoch 999: %.3e (heading to zero, still positive)' % lrs[999])
assert abs(lrs[0] - 2e-5) < 1e-9 and abs(lrs[49] - 1e-3) < 1e-9 and abs(lrs[50] - 1e-3) < 1e-9

## Plot the two-phase curve

In [ ]:
from course_utils.viz import plot_lr_curve
ax, _ = plot_lr_curve(max_lr=MAX_LR, warmup_epochs=WARMUP, total_epochs=TOTAL)
ax.figure.tight_layout()
plt.show()

## How the trainer wires it (read-only)

A custom trainer swaps the scheduler at the right epoch and reuses the optimizer, so momentum carries across the switch. Both stages train the whole network; `'train'` is just the name of the post-warm-up decay phase.

```python
def on_train_epoch_start(self):
    if self.current_epoch == 0:
        self.optimizer, self.lr_scheduler = self.configure_optimizers('warmup_all')
    elif self.current_epoch == self.warmup_duration_whole_net:   # == 50
        self.optimizer, self.lr_scheduler = self.configure_optimizers('train')
    super().on_train_epoch_start()
```

## Compare on a real run

The schedule only matters once you actually train. The comparison that isolates the warm-up is **plain 1e-3 (no warm-up) versus warm-up to 1e-3** (same 1e-3 peak, the only difference is the 50-epoch ramp). Run both fine-tuning jobs, export the validation Dice (`val/ema_fg_dice`, the smoothed foreground Dice nnU-Net uses to pick the best checkpoint) from W&B, save it to `assets/precomputed/real_training_curves.csv` (columns `epoch,plain,warmup`), and the cell below plots the comparison. Higher is better; no curve is fabricated. The curve below is an in-house reproduction on a separate dataset, distinct from the published TBI numbers.

For the evidence table, paste the screenshot of Table 2 from the TBI paper (`arXiv:2504.06741`) onto the slide. For reference, the 5-fold average Dice is: from scratch 53.44, fine-tune at plain 1e-3 53.80, warm-up to 1e-2 53.28, warm-up to 1e-3 54.21. So warm-up to 1e-3 beats plain 1e-3 by 0.41 Dice (modest but real), while warm-up to 1e-2 is worse than not warming up at all.

In [ ]:
import csv
p = PRE / 'real_training_curves.csv'
if p.exists():
    rows = list(csv.DictReader(p.open()))
    ep = [int(r['epoch']) for r in rows]
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.plot(ep, [float(r['plain']) for r in rows], label='plain 1e-3 (no warm-up)')
    ax.plot(ep, [float(r['warmup']) for r in rows], label='warm-up to 1e-3')
    ax.axvline(WARMUP, color='crimson', ls='--', lw=1, label='warm-up ends')
    ax.set_xlabel('epoch'); ax.set_ylabel('validation Dice (EMA)'); ax.legend()
    ax.set_title('Fine-tuning: plain 1e-3 vs warm-up to 1e-3'); fig.tight_layout()
    plt.show()
else:
    print('No real_training_curves.csv yet. Add it after the two fine-tuning runs.')

## A note on generality

The warm-up benefit is task dependent. PANTHER (`arXiv:2508.21775`) tried warm-up and cosine schedules for a pancreas-MRI task and found they did not beat the default poly schedule. Run the small ablation for your own task before assuming warm-up helps.

## Recap
1. Two phases means warm-up then decay of the learning rate, with one peak at epoch 50.
2. The peak is 1e-3, not the default 1e-2.
3. The whole network trains throughout.